## 3.4 Tensor 运算 - 张量的形状操作

#### 1. 形状的核心概念：
shape 描述的是张量每个维度的大小。
* x.shape = (3,) ：1D 向量，有 3 个元素
* x.shape = (2, 3)：2 行 3 列矩阵
* x.shape = (B, C, H, W)：深度学习常见 4D 图像张量

维度（dim）：
**dim（维度） 可以理解为shape的下标**
假设有一个形状为 (3, 1, 5) 的张量：
* dim = 0：第一维，大小为3
* dim=1: 第二维，大小为 1
* dim=2: 第三维，大小为 5


维度大小（ndim）：
**ndim（维度数） = shape 的长度**
* (3,) → ndim = 1
* (2,3) → ndim = 2
* (2,3,4) → ndim = 3

#### 2. 形状操作改变的是什么？
**形状操作大多数时候只是在“改你看数据的方式”，不是改数据本身**
* 你没有把数据重新算一遍
* 只是将数据重新排版显示一遍
* 没有改变内存中的数据结构

#### 3. 重塑形状：reshape vs view（最核心）🔧

##### 3.1 reshape - 最常用
* 只要元素的总个数不变，就可以reshape
* **如果不是连续内存，reshape 会自动复制一份**，然后再修改形状

In [2]:
import torch

a = torch.arange(12) # 创建一个包含0到11的1维张量
print("Bufore reshape:", a)
a_reshaped = a.reshape(3,4) # 将1维张量重塑为3行4列的2维张量
print("After reshape:", a_reshaped)

Bufore reshape: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
After reshape: tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


##### 3.2 view - 更严格的方法
* 要求数据结构必须要是 **contiguous(连续内存)**
* 否则会报错

In [2]:
b = torch.arange(12)
b_view = b.view(3,4) # view() 函数与 reshape() 类似，但它返回的是原始张量的一个视图，而不是一个新的张量。修改 b_view 也会修改 b。
print("After view:", b_view)

After view: tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


##### 3.3 -1 自动推断维度信息
* 由于改变形状时，需要元素的总个数比配，所以可以使用 -1 自动进行维度推断
* 但是只能存在一个 -1 参数

In [3]:
c = torch.arange(12)
c_reshaped = c.reshape(-1,4) # reshape() 函数中的-1表示自动推断该维度的大小，这里会根据总元素数量和其他维度的大小来计算出该维度的大小。由于 c 中有12个元素，且每行有4个元素，因此 reshape(-1,4) 会将 c 重塑为3行4列的2维张量。
print("After reshape with -1:", c_reshaped)

After reshape with -1: tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


#### 4. 加维/压维：unsqueeze / squeeze（批处理必用）📌

##### 4.1 unsqueeze 升高维度：
* 将张量在指定的维度增加维度
* 参数 dim 表示需要操作的具体维度

用途：
1. 将单样本变成 batch = 1 的批处理
2. 为了广播

In [6]:
d = torch.randint(low=0, high=10, size=(3,4)) # 创建一个3行4列的张量，元素值在0到9之间随机生成
print("Original tensor shape:\n", d.shape)

# 在 0维度处，增加一个维度，使得张量的形状变为 (1, 3, 4)
d_unsqueezed = d.unsqueeze(dim=0)
print("After unsqueeze at dim=0:\n", d_unsqueezed.shape)

# 在 1 维度处，增加一个维度，使得张量的形状变为 (3, 1, 4)
d_unsqueezed = d.unsqueeze(dim=1)
print("After unsqueeze at dim=1:\n", d_unsqueezed.shape)

Original tensor shape:
 torch.Size([3, 4])
After unsqueeze at dim=0:
 torch.Size([1, 3, 4])
After unsqueeze at dim=1:
 torch.Size([3, 1, 4])


##### 4.2 squeeze 压缩维度
* 删除 ndim = 1 的某个维度
* 参数为 dim，表示需要删除的具体维度，注意 dim 所在的 ndim 只能为1
* 不传递 dim 参数表示删除所有 ndim = 1 的维度

In [7]:
f = torch.randint(low=0, high=10, size=(1,3,1,5)) # 创建一个形状为 (1, 3, 1, 5) 的张量，元素值在0到9之间随机生成
print("Original tensor shape:\n", f.shape)

# 删除所有ndim为1的维度，使得张量的形状变为 (3, 5)
f_squeezed = f.squeeze()
print("After squeeze:\n", f_squeezed.shape)

# 删除指定维度，比如删除维度0处的维度，使得张量的形状变为 (3, 1, 5)
f_squeezed = f.squeeze(dim=0)
print("After squeeze at dim=0:\n", f_squeezed.shape)

# 删除指定维度，比如删除维度2处的维度，使得张量的形状变为 (1, 3, 5)
f_squeezed = f.squeeze(dim=2)
print("After squeeze at dim=2:\n", f_squeezed.shape)

Original tensor shape:
 torch.Size([1, 3, 1, 5])
After squeeze:
 torch.Size([3, 5])
After squeeze at dim=0:
 torch.Size([3, 1, 5])
After squeeze at dim=2:
 torch.Size([1, 3, 5])


#### 5. 交换张量内部的维度：transpose/permute

##### 5.1 transpose() 交换张量内部的2个维度

In [8]:
g = torch.randint(low=0, high=10, size=(3,4,5))
print("Original tensor shape:\n", g.shape)
# 交换维度0和1，使得张量的形状变为 (4, 3, 5)
g_tansposed = g.transpose(0,1)
print("After transpose:\n", g_tansposed.shape)
# 交换维度0和2，使得张量的形状变为 (5, 4, 3)
g_tansposed = g.transpose(0,2)
print("After transpose:\n", g_tansposed.shape)

Original tensor shape:
 torch.Size([3, 4, 5])
After transpose:
 torch.Size([4, 3, 5])
After transpose:
 torch.Size([5, 4, 3])


##### 5.3 permute() 重新排列张量内部的所有维度顺序

In [9]:
h = torch.randn(2,4,6,8) # 创建一个形状为 (2, 4, 6, 8) 的张量，元素值为随机生成的正态分布数
print("Original tensor shape:\n", h.shape)
# 将维度重新排列为 (6, 2, 8, 4)，使得张量的形状变为 (6, 2, 8, 4)
h_permuted = h.permute(2, 0, 3, 1)
print("After permute:\n", h_permuted.shape)

Original tensor shape:
 torch.Size([2, 4, 6, 8])
After permute:
 torch.Size([6, 2, 8, 4])


#### 6. flatten：拉平（全连接层前必用）🧻

##### 6.1 全部拉平，降为1D张量

In [3]:
i = torch.randn(2,3,4) 
print("Original tensor shape:\n", i.shape)
print("After flatten:\n", i.flatten().shape) # 将张量展平为1维，输出形状为 (24,)

Original tensor shape:
 torch.Size([2, 3, 4])
After flatten:
 torch.Size([24])


##### 6.2 从某个维度之后才开始全部拉平

In [7]:
j = torch.randn(2,2,3,4,5,6,6)
print("Original tensor shape:\n", j.shape)
# 从第1维开始展平，直到第4维，使得张量的形状变为 (2, 2, 60)
j_flattened = j.flatten(start_dim=2)
print("After flatten from dim=2 to dim=4:\n", j_flattened.shape)
# 从第2维开始，到第3维结束
j_flattened_2 = j.flatten(start_dim=3, end_dim=4)
print("After flatten from dim=3 to dim=4:\n", j_flattened_2.shape)

Original tensor shape:
 torch.Size([2, 2, 3, 4, 5, 6, 6])
After flatten from dim=2 to dim=4:
 torch.Size([2, 2, 2160])
After flatten from dim=3 to dim=4:
 torch.Size([2, 2, 3, 20, 6, 6])


#### 7. expand / repeat 扩展某个维度的大小

##### 7.1 expand: 不复制数据，考广播视图扩展
1. 只能扩展


In [8]:
k = torch.randn(2,3,1)
print(k) # 输出原始张量的值
print("Original tensor shape:\n", k.shape)
# 使用 expand() 函数将张量扩展为 (2, 3, 4)，其中新维度的大小为4，原始数据会在新维度上重复
k_expanded = k.expand(2, 3, 4)
print(k_expanded) # 输出扩展后的张量的值
print("After expand:\n", k_expanded.shape)

tensor([[[ 0.3822],
         [ 1.1208],
         [ 0.0822]],

        [[-0.8387],
         [ 1.7553],
         [ 0.3036]]])
Original tensor shape:
 torch.Size([2, 3, 1])
tensor([[[ 0.3822,  0.3822,  0.3822,  0.3822],
         [ 1.1208,  1.1208,  1.1208,  1.1208],
         [ 0.0822,  0.0822,  0.0822,  0.0822]],

        [[-0.8387, -0.8387, -0.8387, -0.8387],
         [ 1.7553,  1.7553,  1.7553,  1.7553],
         [ 0.3036,  0.3036,  0.3036,  0.3036]]])
After expand:
 torch.Size([2, 3, 4])


##### 7.2 repeat: 复制数据

In [9]:
l = torch.randn(2,3,1)
print(l) # 输出原始张量的值
print("Original tensor shape:\n", l.shape)
# 使用 repeat() 函数将张量重复为 (2, 3,4)，其中新维度的大小为4，原始数据会在新维度上重复
l_repeated = l.repeat(1, 1, 4)
print(l_repeated) # 输出重复后的张量的值
print("After repeat:\n", l_repeated.shape)

tensor([[[ 1.3548],
         [ 0.1795],
         [ 0.5471]],

        [[ 0.4002],
         [-1.7401],
         [ 0.9699]]])
Original tensor shape:
 torch.Size([2, 3, 1])
tensor([[[ 1.3548,  1.3548,  1.3548,  1.3548],
         [ 0.1795,  0.1795,  0.1795,  0.1795],
         [ 0.5471,  0.5471,  0.5471,  0.5471]],

        [[ 0.4002,  0.4002,  0.4002,  0.4002],
         [-1.7401, -1.7401, -1.7401, -1.7401],
         [ 0.9699,  0.9699,  0.9699,  0.9699]]])
After repeat:
 torch.Size([2, 3, 4])
